# Station data preparation

Edit the configuration cell. This notebook creates one Hydro client and uses live MERGE/Hydro sources when assets need downloading or `refresh=True`. Azure sign-in may be requested.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from cn3s import HydroDischargeSource, StationCalibrationWorkflow
from hydrography import Hydrography, Stations
from utils.hydrology import Hydrology

In [2]:
station_code = "13870000"
start = "2000-01-01"
end = "2026-08-01"
frequency = "M"
refresh = False
data_root = Path("/data/CN3S")
hydro_root = Path("/data/Geospatial_BR_Data/Hidrografia/BHO_2017_5k")

In [3]:
hydrology = Hydrology()
hydrography = Hydrography(
    watersheds_path=hydro_root / "geoft_bho_2017_5k_area_drenagem.parquet",
    reaches_path=hydro_root / "geoft_bho_2017_5k_trecho_drenagem.parquet",
)
stations = Stations(data_root / "Estacoes_Telemetricas.parquet", hydrography)
stations.assign_cobacia()
workflow = StationCalibrationWorkflow(
    stations,
    data_root=data_root,
    discharge_source=HydroDischargeSource(hydrology),
)

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code FVTL3M8YT to authenticate.


In [4]:
workflow.station_details(station_code)

,value
ESTCODIGO,71564480
ESTNOME,LÁBREA
cobacia,492391
dsversao,BHO bho_2017_v_01_05_5k de 2018-06-20
assignment_status,assigned


In [5]:
workflow.prepare_station(station_code, start, end, (frequency,), refresh=refresh)

{'station_code': '13870000',
 'cobacia': '492391',
 'network_version': 'BHO bho_2017_v_01_05_5k de 2018-06-20',
 'area_station_km2': None,
 'area_nuareacont_km2': 226439.33662677667,
 'rain_periods': {'M': {'start': '2000-01-01', 'end': '2026-08-01'}},
 'input_revisions': {'M': '0746457e2b5941ee8e32755cbaf7972e'},
 'rain_source': 'MERGE',
 'discharge_source': 'Hydro Vazao'}

In [6]:
workflow.preparation_report(station_code, frequency)

,value
station_code,13870000
frequency,M
start,2000-01-01
end,2026-08-01
rain_steps,320
observed_steps,317
paired_steps,317
upstream_area_km2,226439.336627
saved_in,/data/CN3S/stations/13870000


## Test CN3S

In [7]:
from cn3s import CN3S, CN3SOptimizer, CN3SParams
from cn3s.objectives import Objectives


In [8]:
import pandas as pd

q = pd.read_parquet("/data/CN3S/stations/13870000/discharge/monthly.parquet")
prec = pd.read_parquet("/data/CN3S/stations/13870000/rain/monthly.parquet")

In [9]:
CN3SParams?

Init signature:
CN3SParams(
    name: 'str' = 'Unnamed',
    area: 'float' = 34334.0,
    r0: 'float' = 350.0,
    cn_i: 'float' = 7.35,
    alfa: 'float' = 0.2,
    beta: 'float' = 0.00211662329536844,
    k0: 'float' = 1.0,
    k1: 'float' = 0.316,
    k2: 'float' = 0.305,
    act: 'int' = 0,
    warmup_steps: 'int' = 3,
) -> None
Docstring:     
Parameters for the CN3S rainfall-runoff model.

Stores both basin descriptors and the six calibration parameters.
All numeric fields default to values calibrated for the Porto Uruaçu basin.
File:           /workspaces/CN3S/src/cn3s/model.py
Type:           type
Subclasses:     

In [59]:
params = CN3SParams(
    name=f"LABREA ({station_code})",
    area=226439,
    warmup_steps=10,
    r0=50,
    cn_i=30,
    alfa=0.82,
    beta=0.0026,
    k0=0.47,
    k1=0.53,
    k2=0.6,
    act=1,
)

model = CN3S(params, freq="M")

model.run(prec["prec"])

Running CN3S model:   0%|          | 0/309 [00:00<?, ?step/s]

In [60]:
model.evaluate(q["Vazao"])

Aligned 306 time steps between model results and observations.
From 2000-12-01 00:00:00 to 2026-05-01 00:00:00.


0.8902026347254053

In [61]:
model.plot()

In [ ]:
x0 = [50, 40, 0.9, 0.002, 0.23, 0.53, 0.52, 1.0]

optim = CN3SOptimizer(
    prec=prec["prec"],
    observed_q=q["Vazao"],
    model=model,
    train_ratio=0.8,
    max_act=2,
    objective=Objectives.PlainNSE,
)

print(f"Aligned record: {len(optim.aligned)} time steps")
print(f"Train: {optim.train_idx} | Test: {len(optim.aligned) - optim.train_idx}")

Aligned record: 317 time steps
Train: 253 | Test: 64


In [20]:
optim._bounds_list.append((0, 2))

In [21]:
optim.optimize(maxiter=100, tol=1e-4, disp=False, workers=-1, x0=x0, polish=False)

Calibration: differential evolution


1 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
2 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
3 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
4 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
5 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
6 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
7 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
8 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
9 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
10 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
11 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
12 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
13 - Train objective: 0.1020 - [50.0, 40.0, 0.9, 0.0, 0.23, 0.53, 0.52, 1.0]
14 - Tra

Process ForkPoolWorker-4:
Process ForkPoolWorker-5:
Process ForkPoolWorker-2:
Process ForkPoolWorker-3:
Process ForkPoolWorker-6:
Process ForkPoolWorker-1:


Calibration interrupted during evolution; recovering best available result
Calibration: final simulation


In [22]:
optim.result

 message: Interrupted during evolution; best available result recovered
 success: False
     fun: 0.09629883036499642
       x: [ 5.001e+01  1.922e+01  8.179e-01  2.634e-03  4.674e-01
            5.307e-01  5.405e-01  1.404e+00]
     nit: 67
    nfev: 8160

In [63]:
model.results

,prec,past_prec,vj,cnv,s,q_up,r1,q_low,r,q_mm,q_m3s,obs_q_m3s
2000-12-01,163.763378,"[153.12164699570815, 69.10280311158799, 51.719...",1.535807,40.435495,374.16098,0.0,136.794591,82.076754,54.717836,82.076754,7170.284792,4311.170323
2001-01-01,174.673149,"[163.76337848712447, 153.12164699570815, 69.10...",1.67736,43.111446,335.170684,0.0,147.294605,88.376763,58.917842,88.376763,7720.658131,6004.983548
2001-02-01,295.919696,"[174.67314914163092, 163.76337848712447, 153.1...",1.772293,44.871483,312.061081,4.551014,213.343243,128.005946,85.337297,132.55696,11580.272157,10261.213929
2001-03-01,261.848679,"[295.9196955472103, 174.67314914163092, 163.76...",2.132154,51.324214,240.893115,13.553282,216.933858,130.160315,86.773543,143.713597,12554.924047,11432.780000
2001-04-01,269.599685,"[261.84867891630904, 295.9196955472103, 174.67...",2.212715,52.72653,227.730925,22.105689,217.945361,130.767216,87.178144,152.872906,13355.087926,11667.851000
...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-01,170.599517,"[222.55733637339057, 151.22337714592274, 312.4...",2.04043,49.709831,256.965326,0.0,164.946579,98.967947,65.978632,98.967947,8645.911669,10474.494839
2026-06-01,46.941758,"[170.59951716738198, 222.55733637339057, 151.2...",1.932517,47.784863,277.54908,0.0,90.857764,54.514658,36.343105,54.514658,4762.440073,NaN
2026-07-01,55.876945,"[46.94175831545064, 170.59951716738198, 222.55...",1.560319,40.903531,366.97328,0.0,65.957886,39.574732,26.383154,39.574732,3457.277263,NaN
2026-08-01,16.820245,"[55.87694474248927, 46.94175831545064, 170.599...",1.408613,37.972922,414.897692,0.0,35.297885,21.178731,14.119154,21.178731,1850.189277,NaN


## Construct final Chart 

In [64]:
from hydro.station import Station

In [68]:
station = Station(station_code, connector=hydrology.conn)

In [70]:
qd = station.get_series("hidro/telemetria", "vazao")
qd

,EstacaoCodigo,NivelConsistencia,MediaDiaria,vazao_data,val,ano,mes,dia,Sistema,HORESTACAO,ORIGEM
Data,,,,,,,,,,,
1967-07-01,13870000.0,2,1.0,1967-07-01,2629.860107,1967.0,7.0,1.0,Hidro,NaN,NaN
1967-07-02,13870000.0,2,1.0,1967-07-02,2546.629883,1967.0,7.0,2.0,Hidro,NaN,NaN
1967-07-03,13870000.0,2,1.0,1967-07-03,2455.729980,1967.0,7.0,3.0,Hidro,NaN,NaN
1967-07-04,13870000.0,2,1.0,1967-07-04,2384.199951,1967.0,7.0,4.0,Hidro,NaN,NaN
1967-07-05,13870000.0,2,1.0,1967-07-05,2313.729980,1967.0,7.0,5.0,Hidro,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-09-27,NaN,1,NaN,NaN,1150.882000,NaN,NaN,NaN,Telemetria,71564480.0,RHN
2026-09-28,NaN,1,NaN,NaN,1130.155000,NaN,NaN,NaN,Telemetria,71564480.0,RHN
2026-09-29,NaN,1,NaN,NaN,1126.925000,NaN,NaN,NaN,Telemetria,71564480.0,RHN


In [127]:
stats = pd.DataFrame(
    {
        "median": qd.groupby([qd.index.month, qd.index.day])["val"].median(),
        "min": qd.groupby([qd.index.month, qd.index.day])["val"].min(),
        "max": qd.groupby([qd.index.month, qd.index.day])["val"].max(),
    },
)

stats = stats.drop((2, 29))
stats.index = pd.date_range(start="2026-01-01", end="2026-12-31")

s2025 = qd.loc["2025"]
s2026 = qd.loc["2026"]
s2024 = qd.loc["2024"]

monthly_mean = qd.groupby(qd.index.month)["val"].median()

In [134]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_scatter(x=stats.index, y=stats["median"], mode="lines", name="Median", opacity=0.1, line={"color": "blue", "dash": "dash"})
fig.add_scatter(x=stats.index, y=stats["min"], mode="lines", name="Min", opacity=0.1, line={"color": "grey", "width": 0})
fig.add_scatter(x=stats.index, y=stats["max"], mode="lines", name="Max", opacity=0.01, line={"color": "grey", "width": 0}, fill="tonexty", fillcolor="rgba(128, 128, 128, 0.10)")
fig.add_scatter(x=stats.index, y=s2025["val"], mode="lines", name="2025", line={"color": "blue", "width": 1})
fig.add_scatter(x=stats.index, y=s2026["val"], mode="lines", name="2026", line={"color": "black"})
fig.add_scatter(x=stats.index, y=s2024["val"], mode="lines", name="2024", line={"color": "green", "width": 1})

monthly_mean = qd.groupby(qd.index.month)["val"].median()

month_edges = pd.date_range("2026-01-01", "2027-01-01", freq="MS")
month_values = monthly_mean.reindex(month_edges[:-1].month).to_numpy()

monthly_line_x = []
monthly_line_y = []
for month_start, month_end, monthly_value in zip(
    month_edges[:-1], month_edges[1:], month_values
):
    monthly_line_x.extend([month_start, month_end, None])
    monthly_line_y.extend([monthly_value, monthly_value, None])

fig.add_scatter(
    x=monthly_line_x,
    y=monthly_line_y,
    mode="lines",
    name="Monthly mean",
    line={"color": "orange", "width": 1, "dash": "dot"},
)

# ...existing code...
fig.update_layout(
    width=1000,
    height=500,
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis={"showline": True, "linecolor": "black", "mirror": True},
    yaxis={"showline": True, "linecolor": "black", "mirror": True},
)
fig.show()

In [ ]:
prec["prec"]

In [112]:
s2024

,EstacaoCodigo,NivelConsistencia,MediaDiaria,vazao_data,val,ano,mes,dia,Sistema,HORESTACAO,ORIGEM
Data,,,,,,,,,,,
2024-01-01,13870000.0,1,1.0,2024-01-01,4232.889160,2024.0,1.0,1.0,Hidro,NaN,NaN
2024-01-02,13870000.0,1,1.0,2024-01-02,4301.686523,2024.0,1.0,2.0,Hidro,NaN,NaN
2024-01-03,13870000.0,1,1.0,2024-01-03,4389.380859,2024.0,1.0,3.0,Hidro,NaN,NaN
2024-01-04,13870000.0,1,1.0,2024-01-04,4484.299805,2024.0,1.0,4.0,Hidro,NaN,NaN
2024-01-05,13870000.0,1,1.0,2024-01-05,4599.188965,2024.0,1.0,5.0,Hidro,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,13870000.0,1,1.0,2024-12-27,5250.699707,2024.0,12.0,27.0,Hidro,NaN,NaN
2024-12-28,13870000.0,1,1.0,2024-12-28,5414.815430,2024.0,12.0,28.0,Hidro,NaN,NaN
2024-12-29,13870000.0,1,1.0,2024-12-29,5684.442383,2024.0,12.0,29.0,Hidro,NaN,NaN


In [80]:
stats

median          min          max
Data Data                                       
1    1     5948.250000  2287.580078  9478.429688
     2     6018.709961  2424.300049  9585.259766
     3     6132.299805  2546.629883  9674.750000
     4     6254.160156  2657.899902  9773.669922
     5     6399.080078  2733.419922  9873.089844
...                ...          ...          ...
12   27    5711.819824  1969.270020  8918.809570
     28    5711.819824  1965.239990  9039.759766
     29    5684.442383  1997.579956  9152.809570
     30    5767.020020  2079.610107  9266.559570
     31    5850.310059  2180.229980  9363.370117

[366 rows x 3 columns]

In [78]:
pd.date_range(start="2024-01-01", end="2026-12-31")

DatetimeIndex(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-04',
               '2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
               '2024-01-09', '2024-01-10',
               ...
               '2026-12-22', '2026-12-23', '2026-12-24', '2026-12-25',
               '2026-12-26', '2026-12-27', '2026-12-28', '2026-12-29',
               '2026-12-30', '2026-12-31'],
              dtype='datetime64[ns]', length=1096, freq='D')